# FOMO — SJSU Headcount Training & Visualization on Combined Scenes

End-to-end notebook that:
1. Clones [fomo-edge-ai/fomo](https://github.com/fomo-edge-ai/fomo) and installs it
2. Downloads and combines the **SJSU Headcount Scene-1, Scene-2, and Scene-3** datasets from HuggingFace (`bdanko/sjsu-headcount-scene-1`, `-2`, `-3`)
3. Trains **FOMO-m** (the lightweight MobileNetV2 point-localization model) for 10 epochs
4. Visualizes the predictions on validation images — predicted person locations as coloured circles overlaid on the original image

## 1 — Environment Setup

In [1]:
!pip install -q fomo-edge-ai==0.0.12

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 10.5 MB/s eta 0:00:00


You must restart the session after running the above

In [1]:

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")


Using device: cuda
  GPU: Tesla T4


## 2 — Download and Combine Datasets

We download the three datasets from HuggingFace (`bdanko/sjsu-headcount-scene-1`, `bdanko/sjsu-headcount-scene-2`, and `bdanko/sjsu-headcount-scene-3`).
We merge their splits (train, validation, test) into a combined directory, prefixing filenames to prevent name conflicts.

In [2]:
import shutil
import zipfile
from pathlib import Path
import yaml
from huggingface_hub import hf_hub_download

HF_REPOS = {
    "scene-1": "bdanko/sjsu-headcount-scene-1",
    "scene-2": "bdanko/sjsu-headcount-scene-2",
    "scene-3": "bdanko/sjsu-headcount-scene-3"
}

RAW_ROOT = Path("raw_datasets")
DATASET_ROOT = Path("sjsu-headcount-combined")

# 1. Download and extract all dataset zips from scratch
for prefix, repo_id in HF_REPOS.items():
    dataset_dir = RAW_ROOT / prefix
    zip_name = f"sjsu-headcount-scene-{prefix.split('-')[-1]}.zip"
    print(f"Downloading {zip_name} from {repo_id}...")
    shutil.rmtree(dataset_dir, ignore_errors=True)
    zip_path = hf_hub_download(
        repo_id=repo_id,
        filename=zip_name,
        repo_type="dataset"
    )
    print(f"Extracting to {dataset_dir}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(dataset_dir)
    print(f"Dataset {prefix} extraction complete")

# Clean target combined dataset folder
shutil.rmtree(DATASET_ROOT, ignore_errors=True)

# 2. Create combined directory structure
for split in ["train", "valid", "test"]:
    (DATASET_ROOT / split / "images").mkdir(parents=True, exist_ok=True)
    (DATASET_ROOT / split / "labels").mkdir(parents=True, exist_ok=True)

# 3. Merge splits and prefix filenames to avoid collisions
print("\nCombining datasets...")
for prefix, repo_id in HF_REPOS.items():
    dataset_dir = RAW_ROOT / prefix

    split_mapping = {
        "train": "train",
        "valid": "valid",
        "test": "test"
    }

    for src_split, dst_split in split_mapping.items():
        src_img_dir = dataset_dir / src_split / "images"
        src_lbl_dir = dataset_dir / src_split / "labels"

        if not src_img_dir.exists() or not src_lbl_dir.exists():
            continue

        dst_img_dir = DATASET_ROOT / dst_split / "images"
        dst_lbl_dir = DATASET_ROOT / dst_split / "labels"

        for img_path in src_img_dir.glob("*"):
            if img_path.suffix.lower() in [".jpg", ".jpeg", ".png"]:
                new_name = f"{prefix}_{img_path.name}"
                shutil.copy2(img_path, dst_img_dir / new_name)

                # Corresponding label file
                lbl_name = f"{img_path.stem}.txt"
                lbl_path = src_lbl_dir / lbl_name
                if lbl_path.exists():
                    shutil.copy2(lbl_path, dst_lbl_dir / f"{prefix}_{lbl_name}")

print("Dataset combination complete.")

# 4. Create data.yaml for combined dataset
data_yaml_path = DATASET_ROOT / "data.yaml"
data_cfg = {
    "path": str(DATASET_ROOT.resolve()),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 1,
    "names": ["person"]
}
data_yaml_path.write_text(yaml.dump(data_cfg, default_flow_style=False))

print(f"\nDataset config:")
print(f"  path : {data_cfg['path']}")
print(f"  train: {data_cfg['train']}")
print(f"  val  : {data_cfg['val']}")
print(f"  nc   : {data_cfg['nc']} ({data_cfg['names']})")

# Quick sanity check — count images and labels
def count_split(split_dir):
    imgs = list(split_dir.rglob("*.jpg")) + list(split_dir.rglob("*.png")) + list(split_dir.rglob("*.jpeg"))
    lbl_dir = split_dir.parent / "labels"
    lbls = list(lbl_dir.rglob("*.txt")) if lbl_dir.exists() else []
    return len(imgs), len(lbls)

train_imgs, train_lbls = count_split(DATASET_ROOT / "train" / "images")
val_imgs, val_lbls     = count_split(DATASET_ROOT / "valid" / "images")
test_imgs, test_lbls   = count_split(DATASET_ROOT / "test" / "images")
print(f"Combined Train : {train_imgs} images, {train_lbls} label files")
print(f"Combined Val   : {val_imgs} images, {val_lbls} label files")
print(f"Combined Test  : {test_imgs} images, {test_lbls} label files")



Extracting to raw_datasets/scene-1...
Dataset scene-1 extraction complete
Extracting to raw_datasets/scene-2...
Dataset scene-2 extraction complete
Extracting to raw_datasets/scene-3...
Dataset scene-3 extraction complete

Combining datasets...
Dataset combination complete.

Dataset config:
  path : /content/sjsu-headcount-combined
  train: train/images
  val  : valid/images
  nc   : 1 (['person'])
Combined Train : 1296 images, 1296 label files
Combined Val   : 370 images, 370 label files
Combined Test  : 187 images, 187 label files


## 3 — Train FOMO-m

FOMO-m uses a **MobileNetV2 backbone (α=0.5)** with a single-pixel detection head.
Input resolution is **192** → output grid **24x24** (8× downsample).

Training uses:
- Adam, lr=3e-4, ReduceLROnPlateau on val F1
- Weighted CrossEntropy (background=1×, foreground=100×)
- No augmentation (plain resize + ImageNet normalisation)
- Val sweep over confidence thresholds every epoch

In [12]:
from pathlib import Path
from fomo import FOMO

EPOCHS     = 100
BATCH      = 16
MODEL_SIZE = "m"            # "s" | "m" | "l"
PROJECT    = "runs/fomo"
RUN_NAME   = f"sjsu_headcount_all_scenes_{MODEL_SIZE}"

model = FOMO(model_path=None, size=MODEL_SIZE, nb_classes=1, device=device)
print(f"Model: FOMO-{MODEL_SIZE}")
print(f"  Input size : {model.input_size}×{model.input_size}")
print(f"  Grid size  : {model.input_size//8}×{model.input_size//8}")
print(f"  Classes    : {model.nb_classes}")

import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)

results = {}
try:
    results = model.train(
        allow_experimental=True,
        data=str(data_yaml_path),
        epochs=EPOCHS,
        batch=BATCH,
        lr0=3e-4,
        eval_interval=1,
        workers=2,
        device=device,
        project=PROJECT,
        name=RUN_NAME,
        exist_ok=True,

        # Early stopping
        patience=12,

        # Regularization & Speedups
        # optimizer="adamw",
        # weight_decay=1e-4,
        # ema=True,
        # amp=True,

        fg_weight=100.0,

        # Data Augmentations
        mosaic_prob=0.0,
        flip_prob=0.0,
        hsv_prob=0.0,
    )
except KeyboardInterrupt:
    print("\nTraining interrupted by user. Generating fallback results dictionary and loading saved weights...")
    save_dir = Path(PROJECT) / RUN_NAME
    results = {
        "save_dir": str(save_dir),
        "best_checkpoint": str(save_dir / "weights" / "best.pt"),
        "last_checkpoint": str(save_dir / "weights" / "last.pt"),
    }

    # Reload the best (or last) checkpoint so the wrapper model holds the trained weights
    best_ckpt = Path(results["best_checkpoint"])
    last_ckpt = Path(results["last_checkpoint"])
    reload_path = best_ckpt if best_ckpt.exists() else last_ckpt
    if reload_path.exists():
        print(f"Reloading weights from: {reload_path}")
        model._load_weights(str(reload_path))
    else:
        print("No saved checkpoint weights found to reload.")

save_dir = Path(results["save_dir"])
print(f"\nTraining complete / Interrupted. Saved to: {save_dir}")


19:38:55  Using device: cuda
19:38:55  Setting up training...
19:38:55  FOMOLoss: nc=1, fg_weight=100.0
19:38:55  FOMOLoss rebuilt with resolved dataset nc=1
19:38:55  FOMO training dataset: 1296 images
19:38:55  Grid size: 24×24 (imgsz=192, downsample=8)
19:38:55  Iterations per epoch: 81 (batch_per_rank=16, world_size=1)
19:38:55  Optimizer: adam
19:38:55    - pg0 (BN): 19 params
19:38:55    - pg1 (Conv, wd=0.0): 20 params
19:38:55    - pg2 (Bias): 20 params
19:38:55  Saving to: runs/fomo/sjsu_headcount_all_scenes_m
19:38:55  Starting training for 100 epochs
19:38:55  Model: FOMO-m
19:38:55  Batch size: 16
19:38:55  Learning rate: 0.0003


Model: FOMO-m
  Input size : 192×192
  Grid size  : 24×24
  Classes    : 1


19:39:01  Epoch 1 - Average loss: 0.4875
19:39:01  Running FOMO validation for epoch 1
19:39:13  Epoch 1 val | loss=0.3349 | F1=0.3720 | P=0.4491 | R=0.3175 | MeanDist=0.570 | thresh=0.90 | nms_r=2 | TP=247 FP=303 FN=531 | LR=0.000300
19:39:13  New best model saved - Epoch 1: metrics/F1=0.3720
19:39:13  Checkpoint saved: runs/fomo/sjsu_headcount_all_scenes_m/weights/last.pt
19:39:20  Epoch 2 - Average loss: 0.2554
19:39:20  Running FOMO validation for epoch 2
19:39:27  Epoch 2 val | loss=0.2117 | F1=0.5494 | P=0.5354 | R=0.5643 | MeanDist=0.516 | thresh=0.90 | nms_r=2 | TP=439 FP=381 FN=339 | LR=0.000300
19:39:27  New best model saved - Epoch 2: metrics/F1=0.5494
19:39:27  Checkpoint saved: runs/fomo/sjsu_headcount_all_scenes_m/weights/last.pt
19:39:34  Epoch 3 - Average loss: 0.1813
19:39:34  Running FOMO validation for epoch 3
19:39:40  Epoch 3 val | loss=0.1699 | F1=0.6297 | P=0.5932 | R=0.6710 | MeanDist=0.498 | thresh=0.90 | nms_r=2 | TP=522 FP=358 FN=256 | LR=0.000299
19:39:41  N


Training interrupted by user. Generating fallback results dictionary and loading saved weights...
Reloading weights from: runs/fomo/sjsu_headcount_all_scenes_m/weights/best.pt

Training complete / Interrupted. Saved to: runs/fomo/sjsu_headcount_all_scenes_m


In [13]:

from fomo import FOMO

weights_dir = save_dir / "weights"
best_pt  = weights_dir / "best.pt"
last_pt  = weights_dir / "last.pt"
ckpt_path = best_pt if best_pt.exists() else last_pt

trained = FOMO(str(ckpt_path), device=device)
print(f"Loaded: {ckpt_path.name}")
print(f"  family : {trained.family}")
print(f"  size   : {trained.size}")
print(f"  imgsz  : {trained.input_size}")
print(f"  nc     : {trained.nb_classes}")


Loaded: best.pt
  family : fomo
  size   : m
  imgsz  : 192
  nc     : 1


In [14]:
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import traceback

def evaluate_counting(trained_model, data_yaml_path, conf_thres=0.9, nms_radius=2, clean_duplicates=True, dist_threshold=0.04):
    """
    Computes counting metrics (MAE, RMSE, MAPE, Accuracy) on the validation set.
    Prints the first error encountered for debugging.
    """
    import yaml
    from fomo.data import load_data_config, get_img_files, img2label_paths

    # 1. Load dataset config and validation files
    data_cfg = load_data_config(str(data_yaml_path))
    data_dir = data_cfg["root"]

    val_path = data_cfg.get("val", "images/val")
    val_img_files = get_img_files(val_path, prefix=data_dir)
    val_label_files = img2label_paths(val_img_files)

    print(f"Loaded {len(val_img_files)} validation images for counting evaluation.")

    gt_counts = []
    pred_counts = []

    for img_path, label_path in tqdm(zip(val_img_files, val_label_files), total=len(val_img_files)):
        # Calculate ground truth count
        gt_points = []
        if Path(label_path).exists():
            with open(label_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        gt_points.append([float(parts[1]), float(parts[2])])

        # Clean duplicates (points closer than dist_threshold in normalized coords)
        if clean_duplicates and len(gt_points) > 0:
            cleaned_gt = []
            for p in gt_points:
                if not cleaned_gt:
                    cleaned_gt.append(p)
                    continue
                dists = np.linalg.norm(np.array(cleaned_gt) - p, axis=1)
                if np.all(dists > dist_threshold):
                    cleaned_gt.append(p)
            gt_count = len(cleaned_gt)
        else:
            gt_count = len(gt_points)

        # Run prediction
        try:
            pil_img = Image.open(img_path)
            result = trained_model.predict(pil_img, conf=conf_thres, nms_radius=nms_radius)
            pred_count = len(result.points) if result.points is not None else 0
        except Exception as e:
            print(f"\n[ERROR ON IMAGE {img_path}]: {e}")
            traceback.print_exc()
            break  # Stop immediately so we can inspect the traceback

        gt_counts.append(gt_count)
        pred_counts.append(pred_count)

    if len(gt_counts) == 0:
        print("\nNo images were successfully processed. See the error above.")
        return [], []

    gt_counts = np.array(gt_counts)
    pred_counts = np.array(pred_counts)

    # Compute counting metrics
    errors = pred_counts - gt_counts
    abs_errors = np.abs(errors)

    mae = np.mean(abs_errors)
    rmse = np.sqrt(np.mean(errors ** 2))

    non_zero = gt_counts > 0
    if np.sum(non_zero) > 0:
        mape = np.mean(abs_errors[non_zero] / gt_counts[non_zero])
        accuracy = max(0.0, 100.0 * (1 - mape))
    else:
        mape = 0.0
        accuracy = 100.0

    total_gt = np.sum(gt_counts)
    total_pred = np.sum(pred_counts)

    print("\n" + "="*60)
    print("             PRODUCTION COUNTING METRICS REPORT")
    print(f"             (Duplicate Cleaning: {clean_duplicates})")
    print("="*60)
    print(f"Total Validation Images      : {len(gt_counts)}")
    print(f"Total Ground-Truth People    : {total_gt}")
    print(f"Total Predicted People       : {total_pred} (Diff: {total_pred - total_gt:+.0f})")
    print(f"Mean Absolute Error (MAE)    : {mae:.3f} people/image")
    print(f"Root Mean Squared Error (RMSE): {rmse:.3f}")
    print(f"Mean Abs % Error (MAPE)      : {mape*100:.2f}%")
    print(f"Overall Counting Accuracy    : {accuracy:.2f}%")
    print("="*60)

    return gt_counts, pred_counts

# Run evaluation on your best model
gt, pred = evaluate_counting(trained, data_yaml_path, conf_thres=0.9, nms_radius=2, clean_duplicates=True)


Loaded 370 validation images for counting evaluation.


100%|██████████| 370/370 [00:04<00:00, 89.04it/s]


             PRODUCTION COUNTING METRICS REPORT
             (Duplicate Cleaning: True)
Total Validation Images      : 370
Total Ground-Truth People    : 764
Total Predicted People       : 668 (Diff: -96)
Mean Absolute Error (MAE)    : 0.524 people/image
Root Mean Squared Error (RMSE): 0.822
Mean Abs % Error (MAPE)      : 25.39%
Overall Counting Accuracy    : 74.61%


## 6 — Export to TFLite (FP32)

We can now export the trained PyTorch model to a TFLite FP32 flatbuffer using the `export` API.

In [15]:

fp32_path = trained.export(output_path=str(weights_dir / f"{RUN_NAME}_fp32.tflite"))
print(f"FP32 TFLite model exported to: {fp32_path}")


19:47:01  Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
19:47:01  Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
19:47:02  Exporting to TFLite FP32: runs/fomo/sjsu_headcount_all_scenes_m/weights/sjsu_headcount_all_scenes_m_fp32.tflite (input 192x192, batch=1)


(00:00) [START] LiteRT-Torch Convert

(00:00) [START] LiteRT-Torch Convert > Torch Export: serving_default

(00:01) [START] LiteRT-Torch Convert > Torch Export: serving_default > ExportedProgram Run Decompositions

/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


...

(00:09) [ DONE] LiteRT-Torch Convert (+00:09)

(00:00) [START] Write Model to 
runs/fomo/sjsu_headcount_all_scenes_m/weights/sjsu_headcount_all_scenes_m_fp32.tflite

(00:00) [ DONE] Write Model to 
runs/fomo/sjsu_headcount_all_scenes_m/weights/sjsu_headcount_all_scenes_m_fp32.tflite (+00:00)

19:47:11  TFLite FP32 export complete: runs/fomo/sjsu_headcount_all_scenes_m/weights/sjsu_headcount_all_scenes_m_fp32.tflite


FP32 TFLite model exported to: /content/runs/fomo/sjsu_headcount_all_scenes_m/weights/sjsu_headcount_all_scenes_m_fp32.tflite


## 7 — INT8 Quantization

To deploy the model on edge devices (like the Arm Ethos-U NPU), we quantize the weights and activations to INT8.
This uses `INT8Quantizer` with a representative calibration dataset from the validation set.

In [16]:

import numpy as np
from PIL import Image
from torchvision import transforms
from fomo.export import INT8Quantizer, INT8QuantizeConfig

val_img_dir = DATASET_ROOT / "valid" / "images"
calibration_images = sorted(val_img_dir.rglob("*.jpg"))

image_transform = transforms.Compose([
    transforms.Resize((trained.input_size, trained.input_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

def get_calibration_data(image_paths, num_samples=150):
    """Yields representative data formatted for the LiteRT graph."""
    count = 0
    for img_path in image_paths:
        if count >= num_samples:
            return
        img = Image.open(img_path).convert("RGB")
        tensor = image_transform(img).unsqueeze(0).numpy()
        yield {"args_0": tensor}
        count += 1

num_samples = min(len(calibration_images), 150)
print(f"Using {num_samples} validation images for calibration.")
calib_iter = get_calibration_data(calibration_images, num_samples=num_samples)

config = INT8QuantizeConfig(num_calibration_samples=num_samples)
quantizer = INT8Quantizer()

int8_path = quantizer.quantize(
    fp32_tflite=fp32_path,
    calibration_data=calib_iter,
    config=config,
    output_path=str(weights_dir / f"{RUN_NAME}_int8.tflite")
)
print(f"INT8 quantized model exported to: {int8_path}")


19:47:11  Quantizing /content/runs/fomo/sjsu_headcount_all_scenes_m/weights/sjsu_headcount_all_scenes_m_fp32.tflite → runs/fomo/sjsu_headcount_all_scenes_m/weights/sjsu_headcount_all_scenes_m_int8.tflite (samples=150)


Using 150 validation images for calibration.


19:47:12  Calibrating with 150 samples…
/usr/local/lib/python3.12/dist-packages/ai_edge_litert/interpreter.py:472: UserWarning: Warning: Enabling `experimental_preserve_all_tensors` with the BUILTIN or AUTO op resolver is intended for debugging purposes only. Be aware that this can significantly increase memory usage by storing all intermediate tensors. If you encounter memory problems or are not actively debugging, consider disabling this option.
  warnings.warn(
19:47:15  Applying quantization…
19:47:15  Serializing model...
19:47:15  INT8 TFLite export complete: runs/fomo/sjsu_headcount_all_scenes_m/weights/sjsu_headcount_all_scenes_m_int8.tflite


Model name: /content/runs/fomo/sjsu_headcount_all_scenes_m/weights/sjsu_headcount_all_scenes_m_fp32.tflite
Original model size: 130.66 KiB
Quantized model size: 99.76 KiB
Quantization Ratio: 0.76 (1.3x smaller)
Total time: 22.15 ms
INT8 quantized model exported to: /content/runs/fomo/sjsu_headcount_all_scenes_m/weights/sjsu_headcount_all_scenes_m_int8.tflite


## Vela Compilation

In [17]:
!pip install -q ethos-u-vela

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 49.3 MB/s eta 0:00:00


In [18]:
!wget https://raw.githubusercontent.com/bencejdanko/Overhead-People-Counting-YOLOXNano-FOMO-Ethos-U55-NPU/refs/heads/main/configs/default_vela.ini

!vela /content/runs/fomo/sjsu_headcount_all_scenes_m/weights/sjsu_headcount_all_scenes_m_int8.tflite \
  --accelerator-config ethos-u55-256 \
  --config /content/default_vela.ini \
  --memory-mode Shared_Sram \
  --system-config Ethos_U55_High_End_Embedded \
  --output-dir vela_output \
  --optimise Performance


--2026-06-07 19:47:22--  https://raw.githubusercontent.com/bencejdanko/Overhead-People-Counting-YOLOXNano-FOMO-Ethos-U55-NPU/refs/heads/main/configs/default_vela.ini
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 415 [text/plain]
Saving to: ‘default_vela.ini’

default_vela.ini    100%[===================>]     415  --.-KB/s    in 0s      

2026-06-07 19:47:22 (10.3 MB/s) - ‘default_vela.ini’ saved [415/415]


Network summary for sjsu_headcount_all_scenes_m_int8
Accelerator configuration               Ethos_U55_256
System configuration             Ethos_U55_High_End_Embedded
Memory mode                               Shared_Sram
Accelerator clock                                 200 MHz
Design peak SRAM bandwidth                       1.49 GB/s
Design peak Off